# XGBoost Binary Baseline — GenIDS-CIC18

This notebook trains the XGBoost binary baseline on **GenIDS-CIC18** and
evaluates it in the intradomain hold-out partition and in the two interdomain
generalization scenarios: **GenIDS-NB15** and **GenIDS-CIC17**.

The notebook preserves the experimental split, model hyperparameters, feature
standardization, and random seed used in the original baseline.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_curve
from xgboost import XGBClassifier

## 2. Configuration

In [ ]:
# Set this directory to the location of the downloaded GenIDS datasets.
DATA_DIR = Path("../../../data").expanduser().resolve()

DATASET_FILES = {
    "GenIDS-CIC17": "GenIDS-CIC17_nfstream_binario.csv",
    "GenIDS-CIC18": "GenIDS-CIC18_nfstream_binario.csv",
    "GenIDS-NB15": "GenIDS-UNSW15_nfstream_binario.csv",
}

missing_files = [
    DATA_DIR / filename
    for filename in DATASET_FILES.values()
    if not (DATA_DIR / filename).is_file()
]
if missing_files:
    missing = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Dataset files were not found. Update DATA_DIR or DATASET_FILES:\n"
        f"{missing}"
    )

TRAINING_DATASET = "GenIDS-CIC18"
TEST_DATASETS = ["GenIDS-NB15", "GenIDS-CIC17"]
TARGET_COLUMN = "binary"
TEST_SIZE = 0.80
RANDOM_STATE = 42

## 3. Dataset loading and inspection

In [ ]:
datasets = {
    name: pd.read_csv(DATA_DIR / filename, low_memory=False)
    for name, filename in DATASET_FILES.items()
}

summary = []
for name, dataframe in datasets.items():
    summary.append(
        {
            "dataset": name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "class_distribution": (dataframe[TARGET_COLUMN] if TARGET_COLUMN in dataframe else dataframe["label"]).value_counts().to_dict(),
        }
    )

pd.DataFrame(summary)

## 4. Preprocessing functions

In [ ]:
METADATA_COLUMNS = {
    "binary",
    "date",
    "hours",
    "expiration_id",
    "label",
    "mapped_label",
    "Timestamp",
    "src_ip",
    "src_mac",
    "src_oui",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "ip_version",
    "vlan_id",
    "tunnel_id",
}
CATEGORICAL_COLUMNS = ["application_name", "application_category_name"]


def encode_shared_categories(dataframes, columns):
    encoded = {name: frame.copy() for name, frame in dataframes.items()}
    for column in columns:
        available = [frame[column].astype(str) for frame in encoded.values() if column in frame]
        if not available:
            continue
        categories = sorted(pd.concat(available, ignore_index=True).unique())
        mapping = {value: index for index, value in enumerate(categories)}
        for frame in encoded.values():
            if column in frame:
                frame[column] = frame[column].astype(str).map(mapping).astype(np.int64)
    return encoded


def encode_target(dataframes, target_column):
    encoded = {name: frame.copy() for name, frame in dataframes.items()}
    values = pd.concat(
        [frame[target_column] for frame in encoded.values()], ignore_index=True
    )
    if pd.api.types.is_numeric_dtype(values):
        classes = sorted(values.dropna().unique().tolist())
        mapping = {value: int(value) for value in classes}
    else:
        classes = sorted(values.dropna().astype(str).unique().tolist())
        mapping = {value: index for index, value in enumerate(classes)}
        for frame in encoded.values():
            frame[target_column] = frame[target_column].astype(str)
    for frame in encoded.values():
        frame[target_column] = frame[target_column].map(mapping).astype(np.int64)
    return encoded, mapping


def prepare_datasets(dataframes, target_column):
    normalized = {name: frame.copy() for name, frame in dataframes.items()}
    for frame in normalized.values():
        if target_column not in frame and "label" in frame:
            frame[target_column] = frame["label"]
    prepared, target_mapping = encode_target(normalized, target_column)
    prepared = encode_shared_categories(prepared, CATEGORICAL_COLUMNS)

    features = {}
    targets = {}
    reference_columns = None
    for name, frame in prepared.items():
        drop_columns = [
            column for column in METADATA_COLUMNS
            if column != target_column and column in frame.columns
        ]
        frame = frame.drop(columns=drop_columns)
        targets[name] = frame.pop(target_column).astype(np.int64)
        frame = frame.apply(pd.to_numeric, errors="raise").astype(np.float64)
        if not np.isfinite(frame.to_numpy()).all():
            raise ValueError(f"{name} contains NaN or infinite feature values.")
        if reference_columns is None:
            reference_columns = frame.columns.tolist()
        elif set(frame.columns) != set(reference_columns):
            missing = sorted(set(reference_columns) - set(frame.columns))
            extra = sorted(set(frame.columns) - set(reference_columns))
            raise ValueError(
                f"Feature mismatch in {name}. Missing: {missing}; extra: {extra}"
            )
        features[name] = frame.loc[:, reference_columns]

    return features, targets, target_mapping

## 5. Feature preparation and data split

In [ ]:
features, targets, target_mapping = prepare_datasets(datasets, TARGET_COLUMN)
print("Target mapping:", target_mapping)

X_source = features[TRAINING_DATASET]
y_source = targets[TRAINING_DATASET]
X_train, X_test, y_train, y_test = train_test_split(
    X_source,
    y_source,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_interdomain_scaled = {
    name: scaler.transform(features[name]) for name in TEST_DATASETS
}

print(f"Training samples: {len(X_train):,}")
print(f"Intradomain test samples: {len(X_test):,}")
for name in TEST_DATASETS:
    print(f"{name} interdomain samples: {len(features[name]):,}")

## 6. Model training

In [ ]:
model = XGBClassifier(
    eval_metric="logloss",
    n_estimators=300,
    max_depth=10,
    objective="binary:logistic",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_train_scaled, y_train)

## 7. Feature importance

In [ ]:
feature_importance = (
    pd.DataFrame(
        {"Feature": X_train.columns, "Importance": model.feature_importances_}
    )
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)
display(feature_importance.head(10))

top_features = feature_importance.head(10).sort_values("Importance")
ax = top_features.plot.barh(
    x="Feature", y="Importance", figsize=(10, 6), legend=False, color="#2f6da3"
)
ax.set_title(f"Top 10 XGBoost features — {TRAINING_DATASET}")
ax.set_xlabel("Feature importance")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 8. Evaluation function

In [ ]:
def evaluate_binary_xgboost(model, X, y, scenario_name):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    confusion = confusion_matrix(y, predictions, labels=[0, 1])
    tn, fp, fn, tp = confusion.ravel()
    far = fp / (fp + tn) if (fp + tn) else 0.0

    metrics = {
        "Scenario": scenario_name,
        "Accuracy": accuracy_score(y, predictions),
        "Precision (macro)": precision_score(
            y, predictions, average="macro", zero_division=0
        ),
        "Recall (macro)": recall_score(
            y, predictions, average="macro", zero_division=0
        ),
        "F1-score (macro)": f1_score(
            y, predictions, average="macro", zero_division=0
        ),
        "AUC-ROC": roc_auc_score(y, probabilities),
        "Average precision": average_precision_score(y, probabilities),
        "FAR": far,
    }

    print(f"\n{scenario_name}")
    print(classification_report(y, predictions, labels=[0, 1], zero_division=0, digits=4))

    ConfusionMatrixDisplay(confusion_matrix=confusion, display_labels=[0, 1]).plot(
        values_format="d", cmap="Blues"
    )
    plt.title(f"Confusion matrix — {scenario_name}")
    plt.tight_layout()
    plt.show()

    fpr, tpr, _ = roc_curve(y, probabilities, pos_label=1)
    precision, recall, _ = precision_recall_curve(y, probabilities, pos_label=1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(fpr, tpr, label=f"AUC = {metrics['AUC-ROC']:.4f}")
    axes[0].plot([0, 1], [0, 1], "k--")
    axes[0].set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate")
    axes[0].legend()
    axes[1].plot(recall, precision, label=f"AP = {metrics['Average precision']:.4f}")
    axes[1].set(title="Precision–Recall curve", xlabel="Recall", ylabel="Precision")
    axes[1].legend()
    fig.suptitle(scenario_name)
    fig.tight_layout()
    plt.show()

    return metrics

## 9. Intradomain and interdomain evaluation

In [ ]:
results = [
    evaluate_binary_xgboost(
        model, X_test_scaled, y_test, f"{TRAINING_DATASET} (intradomain)"
    )
]
for dataset_name in TEST_DATASETS:
    results.append(
        evaluate_binary_xgboost(
            model,
            X_interdomain_scaled[dataset_name],
            targets[dataset_name],
            f"{TRAINING_DATASET} → {dataset_name}",
        )
    )

results_df = pd.DataFrame(results).set_index("Scenario")
results_df.round(4)